# LC0 / Leela Zero Colab template — small-scale self-play & training

This notebook provides a reproducible template to build Leela Zero (leelaz/autogtp) on Google Colab, run small-scale self-play, and get started with training.

Notes: Colab sessions have time and GPU limits. Use small experiments (10–100 self-play games) for testing. Building leelaz may take 10–30 minutes depending on the runtime.


## 1) Quick environment check
Run these cells first to confirm GPU availability and PyTorch (optional).


In [ ]:
# Check GPU (nvidia-smi may not be present in some runtimes)
!nvidia-smi || echo "No nvidia-smi (no GPU?)"

# Check PyTorch (optional)
try:
    import torch
    print("PyTorch:", torch.__version__, "CUDA available:", torch.cuda.is_available())
except Exception as e:
    print("PyTorch not installed:", e)


## 2) Install system packages and Python deps
These are the common build deps; adjust if a step fails.


In [ ]:
# Install build dependencies (may require a few minutes)
!apt-get update -qq
!apt-get install -y -qq build-essential cmake git libboost-all-dev libssl-dev pkg-config

# Useful Python packages for toy experiments and analysis
!pip install -q numpy python-chess

# Optional: install a CUDA-compatible PyTorch wheel if you need PyTorch for custom training
# Uncomment and choose the correct wheel for the runtime if needed:
# !pip install -q torch torchvision --index-url https://download.pytorch.org/whl/cu118


## 3) Clone and build Leela Zero (leelaz + autogtp)
This tries a CUDA-enabled build. If it fails, try a CPU-only build by removing `-DNN_CUDA=ON`.


In [ ]:
# Clone leela-zero repository
!git clone https://github.com/LeelaChessZero/leela-zero.git
%cd leela-zero
!mkdir -p build
%cd build
# Configure the build (try with CUDA if a GPU and drivers are available)
!cmake .. -DNN_CUDA=ON -DCMAKE_BUILD_TYPE=Release
# Build binaries (may take 10-30 minutes)
!make -j$(nproc)


If the CUDA build fails, try: `!cmake .. -DNN_CUDA=OFF -DCMAKE_BUILD_TYPE=Release` and rebuild.


## 4) Running the engine and generating small self-play games
After a successful build, `leelaz` and `autogtp` should be available in the build directory. Example commands below are a starting point — consult leela-zero docs for full options.


In [ ]:
# Example: run leelaz in GTP mode (replace weights path if you have a network)
%cd /content/leela-zero/build
# Launch engine (background)
!./leelaz --gtp --noponder --threads 1 --gpuid 0 &

# Run autogtp to generate a few self-play games (small test). Flags and binary name may differ.
!./autogtp -g 10 --seed 42 || echo "autogtp not present or flags differ; check build artifacts and README"


## 5) Using pre-trained networks (inference)
Download a pre-trained LC0 network and point `leelaz` at it using `--weights /path/to/net.pb.gz`.
Example networks are available at the Leela Chess Zero project and community mirrors — use small networks for testing.


In [ ]:
# Example: download a network (replace URL below)
!mkdir -p /content/networks
%cd /content/networks
# Replace the URL with a valid LC0 network URL (this is an example placeholder)
# !wget -O net.pb.gz https://lczero.org/networks/example-net.pb.gz
# Then run: ./leelaz --gtp --weights /content/networks/net.pb.gz --threads 1 --gpuid 0


## 6) Small-scale training notes & next steps
- For meaningful networks, run many thousands of self-play games and the official training pipeline — this is computationally expensive.
- Consider starting from a pre-trained network and fine-tuning or running a few hundred self-play games for experiments.
- The official leela-zero documentation and the LCZero community provide detailed instructions for `autogtp` usage and the training server/pool. Links are provided in the repo TRAINING_GUIDE.md.

If helpful, next steps can include:
- Uploading generated games to a remote training pipeline or using the existing LCZero training pool.
- Implementing a small controller script to orchestrate self-play → training → evaluation on Colab.
